# 🧠 Traductor Shiwilu ↔ Español con Evaluación BLEU y Exact Match
Este notebook entrena un modelo `EncoderDecoderModel` con tokenizer subword sobre un dataset sintético
y evalúa la calidad de la traducción con métricas automáticas.

**Incluye:**
- Entrenamiento de tokenizer BPE (`tokenizers`)
- Modelo `EncoderDecoderModel` con `distilbert`
- Evaluación BLEU y Exact Match
- Ejemplos de traducción

In [ ]:
!pip install -U transformers datasets tokenizers sacrebleu protobuf==3.20.3

In [ ]:
# Leer corpus sintético
import pandas as pd
df = pd.read_csv("dataset_sintetico_shiwilu_espanol.csv")
from datasets import Dataset
dataset = Dataset.from_pandas(df).train_test_split(test_size=0.1)

In [ ]:
# Entrenar tokenizer BPE
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

with open("combined_corpus.txt", "w", encoding="utf-8") as f:
    for line in df['shiwilu'].tolist() + df['espanol'].tolist():
        f.write(line + "\n")

tokenizer_bpe = Tokenizer(models.BPE(unk_token='[UNK]'))
tokenizer_bpe.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=500, special_tokens=['[PAD]', '[UNK]', '[BOS]', '[EOS]'])
tokenizer_bpe.train(["combined_corpus.txt"], trainer)
tokenizer_bpe.save("tokenizer.json")

In [ ]:
# Cargar tokenizer
from transformers import PreTrainedTokenizerFast
tok = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer.json",
    bos_token='[BOS]', eos_token='[EOS]', pad_token='[PAD]', unk_token='[UNK]'
)
tok.model_max_length = 128

In [ ]:
# Preprocesamiento
def preprocess(example):
    inputs = tok(example['shiwilu'], padding='max_length', truncation=True, max_length=128)
    targets = tok(example['espanol'], padding='max_length', truncation=True, max_length=128).input_ids
    inputs['labels'] = targets
    return inputs

tokenized_dataset = dataset.map(preprocess, remove_columns=['shiwilu', 'espanol'])

In [ ]:
# Modelo encoder-decoder distilbert
from transformers import EncoderDecoderModel
model = EncoderDecoderModel.from_encoder_decoder_pretrained("distilbert-base-uncased", "distilbert-base-uncased")
model.config.pad_token_id = tok.pad_token_id
model.config.decoder_start_token_id = tok.bos_token_id
model.config.eos_token_id = tok.eos_token_id

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./shiwilu_model",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    predict_with_generate=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tok
)
# Entrena con: trainer.train()

In [ ]:
# Traducción y métricas
from sacrebleu import corpus_bleu

def traducir(texto):
    inputs = tok(texto, return_tensors="pt", padding=True, truncation=True).to(model.device)
    output = model.generate(**inputs, max_length=128)
    return tok.decode(output[0], skip_special_tokens=True)

refs, hyps = [], []
for ex in dataset['test']:
    ref = ex['espanol']
    hyp = traducir(ex['shiwilu'])
    refs.append([ref])
    hyps.append(hyp)

bleu = corpus_bleu(hyps, list(zip(*refs)))
exact = sum([1 for r, h in zip(refs, hyps) if r[0].strip() == h.strip()]) / len(refs)

print(f"BLEU score: {bleu.score:.2f}")
print(f"Exact Match: {exact*100:.2f}%")